# Section 1: Imports

In [4]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)


# Section 2: Load Data

In [3]:
model_df = pd.read_parquet(
    "../data/processed/model_df.parquet"
)

model_df.head()

,customer_unique_id,product_id,product_category_name,price,order_purchase_timestamp
0,861eff4711a542e4b93843c6dd7febb0,a9516a079e37a9c9c36b9b78b10169e8,moveis_escritorio,124.99,2017-05-16 15:05:35
1,290c77bc529b7ac935b93aa66c333dc3,4aa6014eceb682077f9dc4bffebc05b0,utilidades_domesticas,289.00,2018-01-12 20:48:24
2,060e732b5b29e8181a18229c7b0b2b5e,bd07b66896d6f1494f5b86251848ced7,moveis_escritorio,139.94,2018-05-19 16:07:45
3,259dac757896d24d7702b9acbbff3f3c,a5647c44af977b148e0a3a4751a09e2e,moveis_escritorio,149.94,2018-03-13 16:06:38
4,4c93744516667ad3b8f1fb645a3116a4,0be701e03657109a8a4d5168122777fb,esporte_lazer,259.90,2017-09-14 18:14:31


# Section 3: Time-Based Train Test Split

In [5]:
split_date = "2018-06-30"

train_df = model_df[
    model_df["order_purchase_timestamp"] <= split_date
].copy()

test_df = model_df[
    model_df["order_purchase_timestamp"] > split_date
].copy()

print("Train Shape:", train_df.shape)
print("Test Shape :", test_df.shape)

Train Shape: (61419, 5)
Test Shape : (7391, 5)


In [6]:
print("Train Start:", train_df["order_purchase_timestamp"].min())
print("Train End  :", train_df["order_purchase_timestamp"].max())

print("-" * 50)

print("Test Start :", test_df["order_purchase_timestamp"].min())
print("Test End   :", test_df["order_purchase_timestamp"].max())

Train Start: 2016-09-04 21:15:19
Train End  : 2018-06-29 23:09:13
--------------------------------------------------
Test Start : 2018-06-30 00:01:53
Test End   : 2018-08-29 15:00:37


# Section 4: Build Popularity Model

In [7]:
train_popularity = (
    train_df
    .groupby("product_id")
    .size()
    .reset_index(name="purchase_count")
    .sort_values(
        "purchase_count",
        ascending=False
    )
)

train_popularity.head(10)

,product_id,purchase_count
3219,aca2eb7d00ea1a7b8ebd4e68314663af,522
1251,422879e10f46682990de24d770e7f83d,472
2850,99a4788cb24856965c36a24e339b6058,471
1019,368c6c730842d78016ad823897a372db,379
1070,389d119b48cf3043d311335e499d9c6b,368
1584,53759a2ecddad2bb87a079a1f1519f73,357
3903,d1c427060a0f73f6b889a5c7c61f2ac4,324
1589,53b36df67ebb7c41585e8d54d6772e08,311
412,154e7e31ebfa092203795c972e5804a6,276
1171,3dd2a17168ec895c781a9191c1e95ad7,256



# Section 5: Create Normalized Popularity Score


In [8]:
train_popularity["popularity_score"] = (
    train_popularity["purchase_count"]
    /
    train_popularity["purchase_count"].max()
)

train_popularity.head()

,product_id,purchase_count,popularity_score
3219,aca2eb7d00ea1a7b8ebd4e68314663af,522,1.000000
1251,422879e10f46682990de24d770e7f83d,472,0.904215
2850,99a4788cb24856965c36a24e339b6058,471,0.902299
1019,368c6c730842d78016ad823897a372db,379,0.726054
1070,389d119b48cf3043d311335e499d9c6b,368,0.704981



# Section 6: Attach Category Information

In [9]:
product_metadata = (
    train_df[
        [
            "product_id",
            "product_category_name"
        ]
    ]
    .drop_duplicates()
)

train_popularity = (
    train_popularity
    .merge(
        product_metadata,
        on="product_id",
        how="left"
    )
)

train_popularity.head()

,product_id,purchase_count,popularity_score,product_category_name
0,aca2eb7d00ea1a7b8ebd4e68314663af,522,1.000000,moveis_decoracao
1,422879e10f46682990de24d770e7f83d,472,0.904215,ferramentas_jardim
2,99a4788cb24856965c36a24e339b6058,471,0.902299,cama_mesa_banho
3,368c6c730842d78016ad823897a372db,379,0.726054,ferramentas_jardim
4,389d119b48cf3043d311335e499d9c6b,368,0.704981,ferramentas_jardim


# Section 7: Top 10 Products

In [10]:
TOP_K = 10

top_k_products = (
    train_popularity
    .head(TOP_K)
)

top_k_products

,product_id,purchase_count,popularity_score,product_category_name
0,aca2eb7d00ea1a7b8ebd4e68314663af,522,1.000000,moveis_decoracao
1,422879e10f46682990de24d770e7f83d,472,0.904215,ferramentas_jardim
2,99a4788cb24856965c36a24e339b6058,471,0.902299,cama_mesa_banho
3,368c6c730842d78016ad823897a372db,379,0.726054,ferramentas_jardim
4,389d119b48cf3043d311335e499d9c6b,368,0.704981,ferramentas_jardim
5,53759a2ecddad2bb87a079a1f1519f73,357,0.683908,ferramentas_jardim
6,d1c427060a0f73f6b889a5c7c61f2ac4,324,0.620690,informatica_acessorios
7,53b36df67ebb7c41585e8d54d6772e08,311,0.595785,relogios_presentes
8,154e7e31ebfa092203795c972e5804a6,276,0.528736,beleza_saude
9,3dd2a17168ec895c781a9191c1e95ad7,256,0.490421,informatica_acessorios


# Section 8: Save Model Artifact

In [ ]:
train_popularity.to_parquet(
    "..artifacts/popularity_recommender.parquet",
    index=False
)

In [13]:
print(train_df.shape)
print(test_df.shape)
print(top_k_products)

(61419, 5)
(7391, 5)
                         product_id  purchase_count  popularity_score  \
0  aca2eb7d00ea1a7b8ebd4e68314663af             522          1.000000   
1  422879e10f46682990de24d770e7f83d             472          0.904215   
2  99a4788cb24856965c36a24e339b6058             471          0.902299   
3  368c6c730842d78016ad823897a372db             379          0.726054   
4  389d119b48cf3043d311335e499d9c6b             368          0.704981   
5  53759a2ecddad2bb87a079a1f1519f73             357          0.683908   
6  d1c427060a0f73f6b889a5c7c61f2ac4             324          0.620690   
7  53b36df67ebb7c41585e8d54d6772e08             311          0.595785   
8  154e7e31ebfa092203795c972e5804a6             276          0.528736   
9  3dd2a17168ec895c781a9191c1e95ad7             256          0.490421   

    product_category_name  
0        moveis_decoracao  
1      ferramentas_jardim  
2         cama_mesa_banho  
3      ferramentas_jardim  
4      ferramentas_jardim  
5      

# Section 9: Popularity Recommender Evaluation

In [14]:
TOP_K = 10

top_k_product_ids = (
    train_popularity
    .head(TOP_K)["product_id"]
    .tolist()
)

top_k_product_ids

['aca2eb7d00ea1a7b8ebd4e68314663af',
 '422879e10f46682990de24d770e7f83d',
 '99a4788cb24856965c36a24e339b6058',
 '368c6c730842d78016ad823897a372db',
 '389d119b48cf3043d311335e499d9c6b',
 '53759a2ecddad2bb87a079a1f1519f73',
 'd1c427060a0f73f6b889a5c7c61f2ac4',
 '53b36df67ebb7c41585e8d54d6772e08',
 '154e7e31ebfa092203795c972e5804a6',
 '3dd2a17168ec895c781a9191c1e95ad7']

## Evaluate Against Test Set

In [15]:
test_hits = (
    test_df["product_id"]
    .isin(top_k_product_ids)
    .sum()
)

total_test_interactions = len(test_df)

precision_at_k = (
    test_hits / total_test_interactions
)

print(f"Hits: {test_hits}")
print(f"Total Test Interactions: {total_test_interactions}")
print(f"Precision@{TOP_K}: {precision_at_k:.4f}")

Hits: 137
Total Test Interactions: 7391
Precision@10: 0.0185


In [16]:
catalog_size = train_df["product_id"].nunique()

coverage = (
    TOP_K / catalog_size
)

print(f"Catalog Size: {catalog_size}")
print(f"Coverage: {coverage:.6f}")

Catalog Size: 4743
Coverage: 0.002108


In [17]:
model_1_metrics = {
    "precision_at_10": precision_at_k,
    "coverage": coverage,
    "train_rows": len(train_df),
    "test_rows": len(test_df),
    "catalog_size": catalog_size
}

model_1_metrics

{'precision_at_10': np.float64(0.018536057367068055),
 'coverage': 0.002108370229812355,
 'train_rows': 61419,
 'test_rows': 7391,
 'catalog_size': 4743}

In [18]:
import json

model_1_metrics = {
    "precision_at_10": float(precision_at_k),
    "coverage": float(coverage),
    "train_rows": len(train_df),
    "test_rows": len(test_df)
}

with open(
    "../artifacts/model_1_metrics.json",
    "w"
) as f:
    json.dump(
        model_1_metrics,
        f,
        indent=4
    )